# 🧩 Segmentación RFM con K-Means
> **Caso de negocio:** Crisol · Segmentación de clientes para CRM
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Crisol envía **la misma campaña de email marketing a toda su base de clientes**, sin diferenciar entre clientes frecuentes y de alto valor vs. clientes inactivos hace meses.

**Objetivo:** Segmentar la base de clientes según Recencia (días desde la última compra), Frecuencia (compras recientes) y Monto (gasto total), para diseñar campañas diferenciadas por segmento.

**KPIs de éxito:**
- Silhouette score >= 0.4 (clusters bien separados)
- Segmentos interpretables para el equipo de marketing (Champions, Leales, En riesgo, Perdidos)
- Cobertura: todos los clientes asignados a un segmento

**Algoritmo:** K-Means (no supervisado) sobre variables RFM estandarizadas

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Aprendizaje no supervisado — clustering',
    'features': [
        'recencia_dias       → días desde la última compra',
        'frecuencia_compras  → número de compras en el período de análisis',
        'monto_total_k       → gasto total del cliente (miles de S/.)',
    ],
    'criterios_aceptacion': {
        'Silhouette': '>= 0.40',
        'n_clusters': '4 (Champions, Leales, En riesgo, Perdidos)',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con 4 segmentos naturales (RFM)
# En producción: reemplazar con agregación de transacciones (BigQuery)
N = 1200

perfiles = {
    'champion':  {'rec': (8, 5),    'frec': (18, 4),  'monto': (18, 4)},
    'at_risk':   {'rec': (65, 20),  'frec': (7, 3),   'monto': (7, 3)},
    'lost':      {'rec': (130, 30), 'frec': (2, 1),   'monto': (2, 1)},
    'potential': {'rec': (15, 8),   'frec': (4, 2),   'monto': (5, 2)},
}
probs = [0.20, 0.25, 0.30, 0.25]
segmentos_reales = np.random.choice(list(perfiles.keys()), N, p=probs)

recencia_dias      = np.array([max(1, int(np.random.normal(*perfiles[s]['rec'])))   for s in segmentos_reales])
frecuencia_compras = np.array([max(1, int(np.random.normal(*perfiles[s]['frec'])))  for s in segmentos_reales])
monto_total_k      = np.array([max(0.5, round(np.random.normal(*perfiles[s]['monto']), 1)) for s in segmentos_reales])

df = pd.DataFrame({
    'cliente_id':         range(1, N+1),
    'recencia_dias':      recencia_dias,
    'frecuencia_compras': frecuencia_compras,
    'monto_total_k':      monto_total_k,
})

print(f'Dataset: {df.shape}')
df.describe().round(2)

In [ ]:
# Distribución de las variables RFM
fig = make_subplots(rows=1, cols=3, subplot_titles=['Recencia (días)', 'Frecuencia (compras)', 'Monto (k S/.)'])

fig.add_trace(go.Histogram(x=df['recencia_dias'], marker_color='#1a4c8c', nbinsx=30, showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df['frecuencia_compras'], marker_color='#0d9488', nbinsx=20, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=df['monto_total_k'], marker_color='#7c3aed', nbinsx=20, showlegend=False), row=1, col=3)

fig.update_layout(height=350, title='Distribución de variables RFM',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Relación entre recencia y frecuencia, coloreada por monto (sin clusters aún)
fig = px.scatter(df, x='recencia_dias', y='frecuencia_compras', color='monto_total_k',
                 color_continuous_scale='teal', opacity=0.6,
                 title='Recencia vs Frecuencia (color = monto total)')
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['recencia_dias', 'frecuencia_compras', 'monto_total_k']

X_raw = df[FEATURES].fillna(0).values

# K-Means es sensible a la escala — estandarizar es obligatorio
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Observaciones: {X.shape[0]} | Features: {X.shape[1]}')
print(f'Medias después de escalar: {X.mean(axis=0).round(3)}')
print(f'Desv. estándar después de escalar: {X.std(axis=0).round(3)}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Curva de codo: probar k=2..9 para elegir el número de clusters
elbow = []
for k in range(2, 10):
    km_tmp = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km_tmp.fit(X)
    elbow.append({'k': k, 'inertia': km_tmp.inertia_})

elbow_df = pd.DataFrame(elbow)

fig = go.Figure(go.Scatter(x=elbow_df['k'], y=elbow_df['inertia'],
                            mode='lines+markers',
                            line=dict(color='#1a4c8c', width=2.5),
                            marker=dict(size=8, color='#1a4c8c')))
fig.update_layout(title='Curva de codo — Inercia vs número de clusters',
                  xaxis_title='Número de clusters (k)', yaxis_title='Inercia',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(dtick=1, gridcolor='#f0f0f0'), yaxis=dict(gridcolor='#f0f0f0'))
fig.show()

In [ ]:
# Entrenar K-Means con k=4 (Champions, Leales, En riesgo, Perdidos)
N_CLUSTERS = 4
km = KMeans(n_clusters=N_CLUSTERS, init='k-means++', n_init=20, random_state=42)
labels = km.fit_predict(X)

df['cluster'] = labels
df['cluster_label'] = [f'Cluster {l+1}' for l in labels]

print(f'Inercia final: {km.inertia_:.2f}')
print(f'\nTamaño de clusters:')
print(df['cluster_label'].value_counts())

## 5️⃣ Fase 5 — Evaluation

In [ ]:
sil = silhouette_score(X, labels)
ch  = calinski_harabasz_score(X, labels)
db  = davies_bouldin_score(X, labels)

metrics = {'Silhouette': sil, 'Calinski-Harabasz': ch, 'Davies-Bouldin': db}
criterios = {'Silhouette': 0.40}

print('=== MÉTRICAS DE CLUSTERING ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral >= {umbral})'
    print(f'  {k:20s}: {v:.4f}  {estado}')

In [ ]:
# Perfil medio por cluster (en escala original)
centroids_real = scaler.inverse_transform(km.cluster_centers_)
profiles = pd.DataFrame(centroids_real, columns=FEATURES)
profiles.insert(0, 'cluster', [f'Cluster {i+1}' for i in range(N_CLUSTERS)])
profiles['n'] = pd.Series(labels).value_counts().sort_index().values

colors = ['#1a4c8c', '#0d9488', '#c0392b', '#d97706']
fig = go.Figure()
for i, row in profiles.iterrows():
    vals = [row[c] for c in FEATURES]
    fig.add_trace(go.Bar(name=row['cluster'], x=FEATURES, y=vals,
                          marker_color=colors[i % len(colors)],
                          text=[f'{v:.1f}' for v in vals], textposition='outside'))

fig.update_layout(barmode='group', title='Perfil medio por cluster (RFM)',
                  height=400, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(showgrid=False), yaxis=dict(gridcolor='#f0f0f0'))
fig.show()
display(profiles.round(2))

In [ ]:
# Distribución 3D de los clusters (RFM)
fig = go.Figure()
for i, cluster in enumerate(sorted(df['cluster_label'].unique())):
    sub = df[df['cluster_label'] == cluster]
    fig.add_trace(go.Scatter3d(
        x=sub['recencia_dias'], y=sub['frecuencia_compras'], z=sub['monto_total_k'],
        mode='markers', name=cluster,
        marker=dict(size=3, color=colors[i % len(colors)], opacity=0.6)
    ))

fig.update_layout(
    scene=dict(xaxis_title='Recencia (días)', yaxis_title='Frecuencia', zaxis_title='Monto (k S/.)', bgcolor='white'),
    title='Distribución 3D de clusters (RFM)',
    height=480, paper_bgcolor='white'
)
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Etiquetado de negocio automático: ordenar clusters por valor (recencia baja + frecuencia/monto altos = mejor)
profiles['score'] = (
    -profiles['recencia_dias'].rank()
    + profiles['frecuencia_compras'].rank()
    + profiles['monto_total_k'].rank()
)
profiles_sorted = profiles.sort_values('score', ascending=False).reset_index(drop=True)

etiquetas_negocio = ['Champions', 'Leales', 'En riesgo', 'Perdidos']
profiles_sorted['segmento_negocio'] = etiquetas_negocio[:len(profiles_sorted)]

mapa_segmento = dict(zip(profiles_sorted['cluster'], profiles_sorted['segmento_negocio']))
df['segmento_negocio'] = df['cluster_label'].map(mapa_segmento)

print('Mapeo cluster → segmento de negocio:')
display(profiles_sorted[['cluster', 'n', 'recencia_dias', 'frecuencia_compras', 'monto_total_k', 'segmento_negocio']].round(2))

print('\nDistribución final de segmentos:')
print(df['segmento_negocio'].value_counts())

In [ ]:
# Guardar outputs
df[['cliente_id', 'recencia_dias', 'frecuencia_compras', 'monto_total_k', 'segmento_negocio']].to_csv(
    'clientes_rfm_crisol.csv', index=False)
print('Archivo clientes_rfm_crisol.csv guardado ✓')

import joblib
joblib.dump({'model': km, 'scaler': scaler, 'features': FEATURES, 'mapa_segmento': mapa_segmento},
            'modelo_rfm_crisol.pkl')
print('Modelo modelo_rfm_crisol.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
n_clientes = len(df)
champions = (df['segmento_negocio'] == 'Champions').sum()
perdidos  = (df['segmento_negocio'] == 'Perdidos').sum()
en_riesgo = (df['segmento_negocio'] == 'En riesgo').sum()

print('=== RESUMEN EJECUTIVO ===')
print(f'Clientes segmentados:               {n_clientes:,}')
print(f'Champions (VIP):                    {champions:,} ({champions/n_clientes:.0%}) → programa de fidelización')
print(f'En riesgo:                          {en_riesgo:,} ({en_riesgo/n_clientes:.0%}) → campaña de reactivación')
print(f'Perdidos:                           {perdidos:,} ({perdidos/n_clientes:.0%}) → descuento de reconquista o exclusión')
print(f'Silhouette score:                   {sil:.3f}')
print(f'\nArquitectura de producción:')
print('  Transacciones → BigQuery → recalculo mensual de RFM/clusters → segmentación CRM (emailing)')